In [1]:
import mmcv
from mmengine import Config
from mmengine.runner import set_random_seed
from mmdet.apis import init_detector, inference_detector
from mmdet.registry import VISUALIZERS
from mmdet.structures.det_data_sample import DetDataSample
from mmengine.structures.instance_data import InstanceData

from matplotlib import pyplot as plt
import cv2 
import torch
import torchvision.ops.boxes as bops
import copy
import numpy as np
import os
import time
from natsort import natsorted

import random

Disabling PyTorch because PyTorch >= 2.4 is required but found 2.1.0


In [2]:
# Check Pytorch installation
import torch, torchvision
print("torch version:",torch.__version__, "cuda:",torch.cuda.is_available())

# Check MMDetection installation
import mmdet
print("mmdetection:",mmdet.__version__)

# Check mmcv installation
import mmcv
print("mmcv:",mmcv.__version__)

# Check mmengine installation
import mmengine
print("mmengine:",mmengine.__version__)

torch version: 2.1.0 cuda: True
mmdetection: 3.3.0
mmcv: 2.1.0
mmengine: 0.10.4


In [3]:
## dict to MMDetection InstanceData class
def dict_to_instance_data(instance_dict):
    instance_data = InstanceData()
    for key, value in instance_dict.items():
        setattr(instance_data, key, value)
    return instance_data

## dict to MMDetection DetDataSample class
def dict_to_det_data_sample(data_dict):
    det_data_sample = DetDataSample()
    for key, value in data_dict.items():
        if isinstance(value, dict):
            # If the value is a dictionary, we assume it is an InstanceData
            setattr(det_data_sample, key, dict_to_instance_data(value))
        else:
            setattr(det_data_sample, key, value)
    return det_data_sample

def delete_low_confidence_predictions(result,confidence_score_threshold=0.5):
    result = result.cpu().numpy().to_dict()
    low_conf_indices = []
    ## find which predictions are of low classification/confidence score and keep their index for deletion
    for idx in range(len(result["pred_instances"]["scores"])):
        if result["pred_instances"]["scores"][idx]<confidence_score_threshold:
            low_conf_indices.append(idx)    

    ## delete from all components of the result variable the instances with low classification/confidence score
    result["pred_instances"]["bboxes"] = np.delete(result["pred_instances"]["bboxes"],low_conf_indices, axis=0)
    result["pred_instances"]["scores"] = np.delete(result["pred_instances"]["scores"],low_conf_indices, axis=0)
    result["pred_instances"]["masks"] = np.delete(result["pred_instances"]["masks"],low_conf_indices, axis=0)
    result["pred_instances"]["labels"] = np.delete(result["pred_instances"]["labels"],low_conf_indices, axis=0)

    return dict_to_det_data_sample(result)

def delete_overlapping_with_lower_confidence(result,iou_threshold = 0.9):
    result = result.cpu().numpy().to_dict()
    to_delete = []
    ## iterate through all existing pairs of predictions
    for idx in range(len(result["pred_instances"]["bboxes"])):
        for idy in range(idx+1,len(result["pred_instances"]["bboxes"])):
            ## create the pair of bounding boxes to be examined
            box1 = torch.tensor([result["pred_instances"]["bboxes"][idx]], dtype=torch.float)
            box2 = torch.tensor([result["pred_instances"]["bboxes"][idy]], dtype=torch.float)
            ## calculate iou of bounding boxes pair
            iou = bops.box_iou(box1, box2)
            ## if iou is above a defined threshold the two prediction are referring to the same instance,
            ## so we find the one with the lower classification/confidence score and keep its index to be deleted
            if iou>iou_threshold:
                if result["pred_instances"]["scores"][idx]>result["pred_instances"]["scores"][idy]:
                    to_delete.append(idy)
                else:
                    to_delete.append(idx)
    
    ## delete from all components of the result variable the overlapping instances with classification/confidence score
    result["pred_instances"]["bboxes"] = np.delete(result["pred_instances"]["bboxes"],to_delete, axis=0)
    result["pred_instances"]["scores"] = np.delete(result["pred_instances"]["scores"],to_delete, axis=0)
    result["pred_instances"]["masks"] = np.delete(result["pred_instances"]["masks"],to_delete, axis=0)
    result["pred_instances"]["labels"] = np.delete(result["pred_instances"]["labels"],to_delete, axis=0)

    return dict_to_det_data_sample(result)

In [4]:
project_folder = r"C:/Users/nuway/OneDrive/Desktop/RCNN/Mask_RCNN_Training/"
## Dataset settings
path_to_dataset = project_folder
path_to_test_set = project_folder

## Architecture settings
architecture_selected = "mask_rcnn_swin-t-p4-w7_fpn_ms-crop-3x_coco" #_14_28_4
architectures_config_paths_dict = {
    "mask_rcnn_swin-t-p4-w7_fpn_ms-crop-3x_coco" : project_folder + "mask_rcnn_swin-t-p4-w7_fpn_ms-crop-3x_coco.py"
}

variable_to_find_py_config_file = architecture_selected
architecture_selected += "/"

## Folder structure creation
working_folder = project_folder + "/results/" + architecture_selected
online_pretrained_weights_folder = project_folder + "configs/"
training_outputs = working_folder + "training_outputs/"
test_outputs = working_folder + "test_outputs/"
predicted_images = test_outputs + "predicted_images/"
predicted_numerical_data = test_outputs + "predicted_numerical_data/"
configs_folder = working_folder + "configs_output/"

## Check whether the specified path exists or not
os.makedirs(working_folder,exist_ok=True)
os.makedirs(training_outputs,exist_ok=True)
os.makedirs(configs_folder,exist_ok=True)
os.makedirs(test_outputs,exist_ok=True)
# os.makedirs(predicted_images,exist_ok=True)
# os.makedirs(predicted_numerical_data,exist_ok=True)

architecture_config_file = architectures_config_paths_dict[variable_to_find_py_config_file]
files = os.listdir(online_pretrained_weights_folder)
architecture_pretrained_file = online_pretrained_weights_folder + [x for x in files if x.startswith(variable_to_find_py_config_file) and x.endswith(".pth")][0]

dataset_to_be_used = "small_timelapse"
use_online_pretrained = True
config_save_filename = "custom_config_" + architecture_selected[:-1]
training_validation = True
prediction_images_threshold = 0.5
save_pickle_files = False
if save_pickle_files:
    confidence_filtering_threshold = 0.5

In [5]:
if use_online_pretrained:
    print("Using official pretrained model configuration and weights with transfer learning")
    print("--------------------------------------------------------------------------------")
    #select model config file and pretrained data
    #cfg = Config.fromfile(architecture_config_file)
    cfg = Config.fromfile(r"C:\Users\nuway\OneDrive\Desktop\RCNN\Mask_RCNN_Training\configs\mask_rcnn_swin-t-p4-w7_fpn_ms-crop-3x_coco.py")
    cfg.load_from = architecture_pretrained_file

    # Modify dataset type and path
    cfg.dataset_type = 'COCODataset'

    ##----------------------------------------------------------------------------------------------------------------------------------------
    if dataset_to_be_used=="small_timelapse":
        
        cfg.metainfo = {
            'classes': ('Mushrooms'),
            'palette': [
                    (220, 40, 50)
                ] 
            }

        cfg.data_root = path_to_dataset
        
        ## Timelapse train set
        cfg.train_dataloader.dataset.ann_file =  'Germany_Images/Germany_Images.json' #'Train/Train.json'
        cfg.train_dataloader.dataset.data_prefix.img =  'Germany_Images/' #'Train/'
        cfg.train_dataloader.dataset.data_root = cfg.data_root
        cfg.train_dataloader.dataset.metainfo = cfg.metainfo
        
        ## Timelapse validation set
        cfg.val_dataloader.dataset.ann_file = 'Germany_Images_Test/Germany_Images_Test.json' #'Germany_Val_Images/Germany_Val_Images.json' #'Validate/Validate.json'
        cfg.val_dataloader.dataset.data_prefix.img =  'Germany_Images_Test/' #'Germany_Val_Images/' #'Validate/'
        cfg.val_dataloader.dataset.metainfo = cfg.metainfo
        cfg.val_dataloader.dataset.data_root = cfg.data_root
        cfg.val_evaluator.ann_file = cfg.data_root + cfg.val_dataloader.dataset.ann_file
        cfg.val_evaluator.classwise = True
        
        # ## Timelapse test set
        # cfg.test_dataloader.dataset.ann_file = 'Partial3/Partial3.json'
        # cfg.test_dataloader.dataset.data_prefix.img = 'Partial3/'
        # cfg.test_dataloader.dataset.metainfo = cfg.metainfo  
        # cfg.test_dataloader.dataset.data_root = path_to_test_set
        # cfg.test_evaluator.ann_file = path_to_test_set + cfg.test_dataloader.dataset.ann_file
        # cfg.test_evaluator.classwise = True
        
#
        # ## If no test-set is present
        # cfg.test_dataloader.dataset.ann_file = cfg.val_dataloader.dataset.ann_file 
        # cfg.test_dataloader.dataset.data_prefix.img = cfg.val_dataloader.dataset.data_prefix.img 
        # cfg.test_dataloader.dataset.metainfo = cfg.val_dataloader.dataset.metainfo
        # cfg.test_dataloader.dataset.data_root = cfg.val_dataloader.dataset.data_root
        # cfg.test_evaluator.ann_file = cfg.val_evaluator.ann_file

    else:
        pass
        ### Placeholder for a different future dataset 
            

    ##----------------------------------------------------------------------------------------------------------------------------------------
    
    ### modify number of images per batch
    cfg.train_dataloader.batch_size = 2
    cfg.val_dataloader.batch_size = 2
    cfg.test_dataloader.batch_size = 2
    
    ### keep it 1 for reproducability, increase if enough hardware and no need to reproduce
    cfg.train_dataloader.num_workers = 1
    cfg.val_dataloader.num_workers = 1
    cfg.test_dataloader.num_workers = 1
    
    # Set up working dir to save files and logs.
    # cfg.work_dir = './tutorial_exps'
    cfg.work_dir = training_outputs

    # The original learning rate (LR) is set for 8-GPU training.
    # We divide it by 8 since we only use one GPU, if needed chnage it further.
    total_epochs = 20
    cfg["optim_wrapper"]["optimizer"]["lr"] = 0.0001 #0.02 / 8
    cfg["param_scheduler"][1]["end"] = total_epochs
    cfg["param_scheduler"][1]["milestones"] = [total_epochs]
    
    
    cfg["default_hooks"]["logger"]["interval"] = total_epochs
    
    cfg["default_hooks"]["checkpoint"]["save_best"] = "coco/segm_mAP"
    cfg["default_hooks"]["checkpoint"]["rule"] = "greater"
    del cfg["default_hooks"]["checkpoint"]["interval"]
    
    cfg["train_cfg"]["max_epochs"] = total_epochs
    cfg["train_cfg"]["val_interval"] = 2
    
    #### experiment with micro-changes in Mask R-CNN, such as the branch CNN
    # cfg["model"]["roi_head"]["mask_roi_extractor"]["roi_layer"]["output_size"] = 42
    # cfg["model"]["train_cfg"]["rcnn"]["mask_size"] = 84
    # cfg["model"]["roi_head"]["mask_roi_extractor"]["roi_layer"]["output_size"] = 28
    # cfg["model"]["train_cfg"]["rcnn"]["mask_size"] = 56
    cfg["model"]["roi_head"]["mask_roi_extractor"]["roi_layer"]["output_size"] = 14
     
    cfg["model"]["train_cfg"]["rcnn"]["mask_size"] = 28
    # cfg["model"]["roi_head"]["mask_head"]["num_convs"] = 6
    
    
    cfg["default_hooks"]["visualization"]["score_thr"] = 0.5

    # Set the seed for all relevant libraries for reproducibility
    seed = 1
    cfg.seed = seed
    set_random_seed(seed, deterministic=True)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    
    # Ensure deterministic behavior for cuDNN
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

    cfg.gpu_ids = range(1)

    
    del cfg["param_scheduler"][0]


    print(cfg.keys())
    print("-------------------")
    # cfg["device"] = 'cpu'
    cfg["device"] = 'cuda'
    # cuda
    print(cfg.keys())
    print("-------------------")
    

    #for timelapse define resize to match the initial ratio
    resize_w = 1066
    resize_h = 600
    # MAKE COMPATIBLE WITH THE 3-CLASS PROBLEM WE FACE
    ## FOR MASK R-CNN 
    cfg.model.roi_head.bbox_head.num_classes = 3
    cfg.model.roi_head.mask_head.num_classes = 3
    if architecture_selected.endswith("3x_coco/"):
        # cfg.fp16 = dict(loss_scale=512.)
        # cfg.runner.meta = dict(fp16=cfg.fp16)
        pass
    else:
        cfg["train_pipeline"][2]["scale"] = (resize_w,resize_h)
        cfg["test_pipeline"][1]["img_scale"] = (resize_w,resize_h)
        cfg["train_dataloader"]["dataset"]["pipeline"][2]["scale"] = (resize_w,resize_h)
        cfg["val_dataloader"]["dataset"]["pipeline"][1]["scale"] = (resize_w,resize_h)
        cfg["test_dataloader"]["dataset"]["pipeline"][1]["scale"] = (resize_w,resize_h)   
    
      
    # We can initialize the logger for training and have a look
    # at the final config used for training
    print(f'Config:\n{cfg.pretty_text}')

   
    # SAVE CONFIG FILE TO .PY FILE
    text_file = open(configs_folder + config_save_filename + ".py", "w")
    text_file.write(cfg.dump())
    text_file.close()
    
else:    
    # LOAD CONFIG FILE FROM CUSTOM SAVED .PY FILE AND CHANGE THE PRETRAINED WEIGHT PATH
    architecture_config_file = configs_folder + [x for x in os.listdir(configs_folder) if variable_to_find_py_config_file in x and x.endswith(".py")][0]
    architecture_pretrained_file = training_outputs + [x for x in os.listdir(training_outputs) if x.startswith("best")][0]
    # cfg.work_dir = working_folder
    cfg = Config.fromfile(architecture_config_file) 
    cfg["load_from"] = architecture_pretrained_file

    print("Using custom pretrained model configuration file: ", architecture_config_file)
    print("Using custom pretrained model weights: ", architecture_pretrained_file)
    print("---------------------------------------------------------")    
    
    # print(f'Config:\n{cfg.pretty_text}')


Using official pretrained model configuration and weights with transfer learning
--------------------------------------------------------------------------------
dict_keys(['model', 'dataset_type', 'data_root', 'backend_args', 'train_pipeline', 'test_pipeline', 'train_dataloader', 'val_dataloader', 'test_dataloader', 'val_evaluator', 'test_evaluator', 'train_cfg', 'val_cfg', 'test_cfg', 'param_scheduler', 'optim_wrapper', 'auto_scale_lr', 'default_scope', 'default_hooks', 'env_cfg', 'vis_backends', 'visualizer', 'log_processor', 'log_level', 'load_from', 'resume', 'pretrained', 'max_epochs', 'metainfo', 'work_dir', 'seed', 'gpu_ids'])
-------------------
dict_keys(['model', 'dataset_type', 'data_root', 'backend_args', 'train_pipeline', 'test_pipeline', 'train_dataloader', 'val_dataloader', 'test_dataloader', 'val_evaluator', 'test_evaluator', 'train_cfg', 'val_cfg', 'test_cfg', 'param_scheduler', 'optim_wrapper', 'auto_scale_lr', 'default_scope', 'default_hooks', 'env_cfg', 'vis_backen

In [6]:
path_to_train_py = r"C:/Users/nuway/OneDrive/Desktop/RCNN/tools/train.py"
path_to_config_file = r"C:\Users\nuway\OneDrive\Desktop\RCNN\Mask_RCNN_Training\results\mask_rcnn_swin-t-p4-w7_fpn_ms-crop-3x_coco\configs_output\custom_config_mask_rcnn_swin-t-p4-w7_fpn_ms-crop-3x_coco.py"
!python {path_to_train_py} {path_to_config_file}

03/20 13:09:28 - mmengine - INFO - 
------------------------------------------------------------
System environment:
    sys.platform: win32
    Python: 3.11.9 | packaged by Anaconda, Inc. | (main, Apr 19 2024, 16:40:41) [MSC v.1916 64 bit (AMD64)]
    CUDA available: True
    MUSA available: False
    numpy_random_seed: 1487991688
    GPU 0: NVIDIA GeForce RTX 3060 Laptop GPU
    CUDA_HOME: None
    MSVC: Microsoft (R) C/C++ Optimizing Compiler Version 19.37.32825 for x64
    GCC: n/a
    PyTorch: 2.1.0
    PyTorch compiling details: PyTorch built with:
  - C++ Version: 199711
  - MSVC 192930151
  - Intel(R) Math Kernel Library Version 2020.0.2 Product Build 20200624 for Intel(R) 64 architecture applications
  - Intel(R) MKL-DNN v3.1.1 (Git Hash 64f6bcbcbab628e96f33a62c3e975f8535a7bde4)
  - OpenMP 2019
  - LAPACK is enabled (usually provided by MKL)
  - CPU capability usage: AVX2
  - CUDA Runtime 11.8
  - NVCC architecture flags: -gencode;arch=compute_50,code=sm_50;-gencode;arch=compu

Disabling PyTorch because PyTorch >= 2.4 is required but found 2.1.0
Disabling PyTorch because PyTorch >= 2.4 is required but found 2.1.0
Traceback (most recent call last):
  File "C:\Users\nuway\OneDrive\Desktop\RCNN\tools\train.py", line 121, in <module>
    main()
  File "C:\Users\nuway\OneDrive\Desktop\RCNN\tools\train.py", line 117, in main
    runner.train()
  File "c:\Users\nuway\OneDrive\Desktop\Realsense Project\Python_Marigold\.conda\Lib\site-packages\mmengine\runner\runner.py", line 1777, in train
    model = self.train_loop.run()  # type: ignore
            ^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\nuway\OneDrive\Desktop\Realsense Project\Python_Marigold\.conda\Lib\site-packages\mmengine\runner\loops.py", line 96, in run
    self.run_epoch()
  File "c:\Users\nuway\OneDrive\Desktop\Realsense Project\Python_Marigold\.conda\Lib\site-packages\mmengine\runner\loops.py", line 113, in run_epoch
    self.run_iter(idx, data_batch)
  File "c:\Users\nuway\OneDrive\Desktop\Realsense Proje

In [ ]:
## Create and execute the training command
path_to_test_py = r"C:/Users/nuway/OneDrive/Desktop/RCNN/tools/test.py"
path_to_config_file = r"C:\Users\nuway\OneDrive\Desktop\RCNN\Mask_RCNN_Training\results\mask_rcnn_swin-t-p4-w7_fpn_ms-crop-3x_coco\configs_output\custom_config_mask_rcnn_swin-t-p4-w7_fpn_ms-crop-3x_coco.py"
path_to_best_checkpoint = r"C:\Users\nuway\OneDrive\Desktop\RCNN\Mask_RCNN_Training\results\mask_rcnn_swin-t-p4-w7_fpn_ms-crop-3x_coco\training_outputs\custom_config_mask_rcnn_swin-t-p4-w7_fpn_ms-crop-3x_coco.py"
path_to_images_output_folder = r"C:\Users\nuway\OneDrive\Desktop\RCNN\Mask_RCNN_Training\results\mask_rcnn_swin-t-p4-w7_fpn_ms-crop-3x_coco\test_outputs\test_predictions"
path_to_outputs = r"C:\Users\nuway\OneDrive\Desktop\RCNN\Mask_RCNN_Training\results\mask_rcnn_swin-t-p4-w7_fpn_ms-crop-3x_coco\test_outputs"
path_to_output_txt = r"C:\Users\nuway\OneDrive\Desktop\RCNN\Mask_RCNN_Training\results\mask_rcnn_swin-t-p4-w7_fpn_ms-crop-3x_coco\test_outputs\output.txt"
path_to_pkl_file = r"C:\Users\nuway\OneDrive\Desktop\RCNN\Mask_RCNN_Training\results\mask_rcnn_swin-t-p4-w7_fpn_ms-crop-3x_coco\test_outputs\results.pkl"

## With and without --show-dir {path_to_images_output_folder} in order to save or not the predicted images as files
# !python {path_to_test_py} {path_to_config_file} {path_to_best_checkpoint} --work-dir {path_to_outputs} --out {path_to_pkl_file} > {path_to_output_txt}
!python {path_to_test_py} {path_to_config_file} {path_to_best_checkpoint} --work-dir {path_to_outputs} --out {path_to_pkl_file} --show-dir {path_to_images_output_folder} > {path_to_output_txt}

In [ ]:
## Create and execute the confusion matrix command
path_to_conf_matrix_py = "/home/scio/mmdetection/tools/analysis_tools/confusion_matrix.py"
!python {path_to_conf_matrix_py} {path_to_config_file}  {path_to_pkl_file}  {path_to_outputs} --show --color-theme coolwarm

In [ ]:
## Create and execute the LOSSES curves command, it is necessery to point to the correct json file that has the training logs
path_to_logs_py = "/home/scio/mmdetection/tools/analysis_tools/analyze_logs.py"
path_to_logs = "/home/scio/christos/Mushrooms/cluster_mushrooms/results/mask_rcnn_swin-s-p4-w7_fpn_fp16_ms-crop-3x_coco/training_outputs/20241104_122931/vis_data/20241104_122931.json"
path_to_save_losses = path_to_outputs + "losses.png"
!python {path_to_logs_py} plot_curve {path_to_logs} --keys loss_cls loss_bbox loss_mask --legend loss_cls loss_bbox loss_mask --out {path_to_save_losses}

In [ ]:
## Create and execute the ACCURACY curve command, it is necessery to point to the correct json file that has the training logs
path_to_logs_py = "/home/scio/mmdetection/tools/analysis_tools/analyze_logs.py"
path_to_logs = "/home/scio/christos/Mushrooms/cluster_mushrooms/results/mask_rcnn_swin-s-p4-w7_fpn_fp16_ms-crop-3x_coco/training_outputs/20241104_122931/vis_data/20241104_122931.json"
path_to_save_accuraccy = path_to_outputs + "accuraccy.png"
!python {path_to_logs_py} plot_curve {path_to_logs} --keys acc --legend acc  --out {path_to_save_accuraccy}

In [ ]:
## STANDALONE Inference on test set
test_set_path = "/home/scio/christos/Mushrooms/cluster_mushrooms/data/1st_test_set/images/"

test_set = natsorted(os.listdir(test_set_path))
test_set = [x for x in test_set if x.endswith("JPG") or x.endswith("jpg")]

architecture_selected = "mask_rcnn_convnext-t_p4_w7_fpn_fp16_ms-crop_3x_coco/"

project_folder = "/home/scio/christos/Mushrooms/"
working_folder = project_folder + "cluster_mushrooms/results/" + architecture_selected
training_outputs = working_folder + "training_outputs/"
configs_folder = working_folder + "configs/"
test_outputs = working_folder + "test_outputs/"
predicted_images = test_outputs + "predicted_images/"

best_checkpoint_weights = [x for x in os.listdir(training_outputs) if x.endswith("pth") and x.startswith("best")][0]

architecture_config_file = configs_folder + [x for x in os.listdir(configs_folder) if architecture_selected[:-1] in x and x.endswith(".py")][0]
model = init_detector(Config.fromfile(architecture_config_file), training_outputs + best_checkpoint_weights, device='cpu')

# init visualizer(run the block only once in jupyter notebook)
visualizer = VISUALIZERS.build(model.cfg.visualizer)
# the dataset_meta is loaded from the checkpoint and
# then pass to the model in init_detector
visualizer.dataset_meta = model.dataset_meta
visualizer.dataset_meta["palette"][0] = (20, 220, 60)
visualizer.dataset_meta["palette"][1] = (220, 40, 50)
visualizer.dataset_meta["palette"][2] = (250, 250, 55)


for test_img in test_set:
    print(test_img)
    img = mmcv.imread(test_set_path + test_img)
    
    save_name = test_img.replace(".JPG", "_prediction.jpg")
    save_name = test_img.replace(".jpg", "_prediction.jpg")

    result = inference_detector(model, img)
    
#     result = delete_low_confidence_predictions(result,confidence_score_threshold=0.5)
#     result = delete_overlapping_with_lower_confidence(result,iou_threshold = 0.7)

#     pickle_save_name = test_img.replace(".jpg", "_data.pkl")
#     pickle_save_name = test_img.replace(".JPG", "_data.pkl")
    
#     masks_to_save = [result.numpy().to_dict()["pred_instances"]["labels"], result.numpy().to_dict()["pred_instances"]["masks"]]
#     save_data_with_pickle(predicted_numerical_data + "/" + pickle_save_name,'masks_to_save')
#     save_data_with_pickle("/media/scio/702AB0B82AB07CA2/christos_mushrooms/1st_timelapse_partA_numerical/" + pickle_save_name,'masks_to_save')
    
#     img = mmcv.image.bgr2rgb(img)
    
    ## show the results
    visualizer.add_datasample(
        'result',
        mmcv.image.bgr2rgb(img),
        data_sample=result,
        draw_gt = None,
        wait_time=0,
        out_file=predicted_images + "prediction_" + test_img,
        pred_score_thr=0.5
    )
    